In [1]:
import json
from pathlib import Path
from collections import Counter

manifest = Path("../data/raw/kitti.ndjson")

records = [
    json.loads(line)
    for line in manifest.read_text(encoding="utf-8").splitlines()
]

dataset_info = next(r for r in records if r["type"] == "dataset")
images = [r for r in records if r["type"] == "image"]

print("Dataset:", dataset_info["name"])
print("Total images:", len(images))
print("Splits:", Counter(img["split"] for img in images))
print("First image:", images[0]["file"])
print("Image dimensions:", images[0]["width"], "x", images[0]["height"])
print("Bounding boxes:", len(images[0]["annotations"]["boxes"]))

Dataset: KITTI
Total images: 7481
Splits: Counter({'train': 5985, 'val': 1496})
First image: 004874.jpg
Image dimensions: 1242 x 375
Bounding boxes: 10


In [2]:
import json
from pathlib import Path
from pprint import pprint

manifest_path = Path("../data/raw/kitti.ndjson")

with manifest_path.open("r", encoding="utf-8") as f:
    first_record = json.loads(next(line for line in f if line.strip()))

print("Top-level keys:")
print(list(first_record.keys()))

print("\nFirst record:")
pprint(first_record, sort_dicts=False)

Top-level keys:
['type', 'task', 'name', 'description', 'bytes', 'url', 'class_names', 'version', 'created_at', 'updated_at']

First record:
{'type': 'dataset',
 'task': 'detect',
 'name': 'KITTI',
 'description': 'Car-mounted camera views of road scenes around Karlsruhe, '
                'Germany, from the KITTI autonomous-driving benchmark (KIT and '
                'TTI Chicago). Recorded with a sensor-equipped station wagon '
                'in city, rural, and highway traffic, with vehicles, '
                'pedestrians, and cyclists labeled under frequent occlusion '
                'and truncation. Ultralytics splits the labeled KITTI training '
                'images into train and val, omits the unlabeled test set, and '
                'drops DontCare regions. Source: Geiger, Lenz, Urtasun, "Are '
                'We Ready for Autonomous Driving? The KITTI Vision Benchmark '
                'Suite," CVPR 2012; cvlibs.net/datasets/kitti.',
 'bytes': 756489412,
 'url': 'ht

In [3]:
import json
from pathlib import Path
from pprint import pprint

manifest_path = Path("../data/raw/kitti.ndjson")

with manifest_path.open("r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

print("Total records:", len(records))
print("\nRecord types:")
for i, record in enumerate(records[:5]):
    print(f"Record {i}: type={record.get('type')}, keys={list(record.keys())}")

print("\nFirst image record:")
image_record = next(
    (record for record in records if record.get("type") != "dataset"),
    None
)

if image_record:
    pprint(image_record, sort_dicts=False)
else:
    print("No image records found in the first scan.")

Total records: 7482

Record types:
Record 0: type=dataset, keys=['type', 'task', 'name', 'description', 'bytes', 'url', 'class_names', 'version', 'created_at', 'updated_at']
Record 1: type=image, keys=['type', 'file', 'url', 'width', 'height', 'split', 'annotations']
Record 2: type=image, keys=['type', 'file', 'url', 'width', 'height', 'split', 'annotations']
Record 3: type=image, keys=['type', 'file', 'url', 'width', 'height', 'split', 'annotations']
Record 4: type=image, keys=['type', 'file', 'url', 'width', 'height', 'split', 'annotations']

First image record:
{'type': 'image',
 'file': '004874.jpg',
 'url': 'https://cdn.ul.run/eu/i/001712f9117b42901313183312742d79.jpg?Expires=1793352600&KeyName=key-v1&Signature=cO5JIQtyZXsFhpBia9kBp3KF1Os',
 'width': 1242,
 'height': 375,
 'split': 'train',
 'annotations': {'boxes': [[0, 0.05621, 0.67723, 0.11247, 0.64019],
                           [0, 0.82953, 0.76617, 0.3393, 0.46226],
                           [0, 0.38128, 0.55116, 0.05461, 

In [4]:
import json
import requests
from pathlib import Path

manifest_path = Path("../data/raw/kitti.ndjson")
test_dir = Path("../data/test")
test_dir.mkdir(parents=True, exist_ok=True)

# Find the first image record without loading the entire manifest
with manifest_path.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        record = json.loads(line)
        if record.get("type") == "image":
            break

print("Filename:", record["file"])
print("Split:", record["split"])
print("Image URL:", record["url"][:150], "...")

try:
    response = requests.get(record["url"], timeout=30)
    print("HTTP status:", response.status_code)
    print("Content type:", response.headers.get("content-type"))
    print("Downloaded bytes:", len(response.content))

    response.raise_for_status()

    image_path = test_dir / record["file"]
    image_path.write_bytes(response.content)

    print("Saved to:", image_path.resolve())

except requests.RequestException as e:
    print("Download failed:", repr(e))

Filename: 004874.jpg
Split: train
Image URL: https://cdn.ul.run/eu/i/001712f9117b42901313183312742d79.jpg?Expires=1793352600&KeyName=key-v1&Signature=cO5JIQtyZXsFhpBia9kBp3KF1Os ...
HTTP status: 200
Content type: image/jpeg
Downloaded bytes: 98665
Saved to: C:\Users\yatha\Desktop\autonomous-driving-perception\data\test\004874.jpg


In [5]:
import json
import time
import requests
from pathlib import Path
from collections import Counter

manifest_path = Path("../data/raw/kitti.ndjson")
dataset_dir = Path("../data/kitti_yolo")

for split in ["train", "val"]:
    (dataset_dir / "images" / split).mkdir(parents=True, exist_ok=True)
    (dataset_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

# Load only image records, not the dataset metadata record
image_records = []
with manifest_path.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        record = json.loads(line)
        if record.get("type") == "image":
            image_records.append(record)

# Select up to 200 images from each split
selected = []
for split in ["train", "val"]:
    split_records = [r for r in image_records if r["split"] == split]
    selected.extend(split_records[:200])

session = requests.Session()
stats = Counter()
failed_files = []

for i, record in enumerate(selected, start=1):
    split = record["split"]
    filename = Path(record["file"]).name
    image_path = dataset_dir / "images" / split / filename
    label_path = dataset_dir / "labels" / split / f"{Path(filename).stem}.txt"

    try:
        # Reuse already downloaded images
        if not image_path.exists():
            response = session.get(record["url"], timeout=30)
            response.raise_for_status()

            if not response.headers.get("content-type", "").startswith("image/"):
                raise ValueError("URL did not return an image")

            image_path.write_bytes(response.content)

        # Expected box format: [class_id, x_center, y_center, width, height]
        annotations = record.get("annotations", {})
        boxes = annotations.get("boxes", [])

        lines = []
        for box in boxes:
            if len(box) != 5:
                continue

            class_id, x_center, y_center, width, height = box
            class_id = int(class_id)

            # Validate normalized YOLO coordinates
            coords = [float(x_center), float(y_center),
                      float(width), float(height)]

            if not all(0 <= x <= 1 for x in coords):
                continue
            if coords[2] <= 0 or coords[3] <= 0:
                continue

            lines.append(
                f"{class_id} " + " ".join(f"{x:.6f}" for x in coords)
            )

        label_path.write_text(
            "\n".join(lines), encoding="utf-8"
        )
        stats[f"{split}_success"] += 1
        stats["labels_written"] += len(lines)

    except Exception as e:
        stats[f"{split}_failed"] += 1
        failed_files.append((filename, str(e)))

    if i % 25 == 0:
        print(f"Processed {i}/{len(selected)} images")

print("\nDownload summary:")
print(dict(stats))
print("Failures:", len(failed_files))
print("First failures:", failed_files[:5])
print("Dataset directory:", dataset_dir.resolve())

Processed 25/400 images
Processed 50/400 images
Processed 75/400 images
Processed 100/400 images
Processed 125/400 images
Processed 150/400 images
Processed 175/400 images
Processed 200/400 images
Processed 225/400 images
Processed 250/400 images
Processed 275/400 images
Processed 300/400 images
Processed 325/400 images
Processed 350/400 images
Processed 375/400 images
Processed 400/400 images

Download summary:
{'train_success': 199, 'labels_written': 2134, 'train_failed': 1, 'val_success': 200}
Failures: 1
First failures: [('002005.jpg', "HTTPSConnectionPool(host='cdn.ul.run', port=443): Read timed out.")]
Dataset directory: C:\Users\yatha\Desktop\autonomous-driving-perception\data\kitti_yolo


In [6]:
from pathlib import Path
from collections import Counter

dataset_dir = Path("../data/kitti_yolo")

for split in ["train", "val"]:
    images = list((dataset_dir / "images" / split).glob("*.jpg"))
    labels = list((dataset_dir / "labels" / split).glob("*.txt"))

    box_count = 0
    for label in labels:
        text = label.read_text(encoding="utf-8").strip()
        if text:
            box_count += len(text.splitlines())

    print(f"\n{split.upper()}")
    print("Images:", len(images))
    print("Label files:", len(labels))
    print("Bounding boxes:", box_count)


TRAIN
Images: 199
Label files: 199
Bounding boxes: 1129

VAL
Images: 200
Label files: 200
Bounding boxes: 1005


In [7]:
import json
from pathlib import Path
from collections import Counter

manifest_path = Path("../data/raw/kitti.ndjson")
dataset_dir = Path("../data/kitti_yolo")

# Read class names from dataset metadata
with manifest_path.open("r", encoding="utf-8") as f:
    metadata = json.loads(next(line for line in f if line.strip()))

class_names = metadata["class_names"]
print("Class mapping:")
print(class_names)

class_counts = Counter()
invalid_lines = []
empty_labels = 0

for split in ["train", "val"]:
    label_dir = dataset_dir / "labels" / split

    for label_file in label_dir.glob("*.txt"):
        lines = label_file.read_text(encoding="utf-8").strip().splitlines()

        if not lines:
            empty_labels += 1
            continue

        for line_number, line in enumerate(lines, start=1):
            parts = line.split()

            try:
                values = [float(x) for x in parts]
                if len(values) != 5:
                    raise ValueError("Expected 5 values")

                class_id = int(values[0])
                coords = values[1:]

                if class_id < 0 or str(class_id) not in class_names:
                    raise ValueError(f"Unknown class ID: {class_id}")

                if not all(0 <= x <= 1 for x in coords):
                    raise ValueError("Coordinates outside [0, 1]")

                if coords[2] <= 0 or coords[3] <= 0:
                    raise ValueError("Non-positive box width/height")

                class_counts[class_id] += 1

            except Exception as e:
                invalid_lines.append(
                    (str(label_file), line_number, line, str(e))
                )

print("\nBounding boxes per class:")
for class_id, count in sorted(class_counts.items()):
    print(f"{class_id} ({class_names[str(class_id)]}): {count}")

print("\nEmpty label files:", empty_labels)
print("Invalid annotation lines:", len(invalid_lines))
print("Examples:", invalid_lines[:5])


Class mapping:
{'0': 'car', '1': 'van', '2': 'truck', '3': 'pedestrian', '4': 'person_sitting', '5': 'cyclist', '6': 'tram', '7': 'misc'}

Bounding boxes per class:
0 (car): 1486
1 (van): 117
2 (truck): 56
3 (pedestrian): 272
4 (person_sitting): 6
5 (cyclist): 125
6 (tram): 27
7 (misc): 45

Empty label files: 0
Invalid annotation lines: 0
Examples: []


In [10]:
from pathlib import Path
from ultralytics import YOLO

project_dir = Path.cwd().parent
data_yaml = project_dir / "data" / "kitti.yaml"

model = YOLO("yolo11n.pt")

results = model.train(
    data=str(data_yaml),
    epochs=3,
    imgsz=416,
    batch=4,
    device=0,
    workers=2,
    project=str(project_dir / "runs"),
    name="kitti_smoke_test",
    exist_ok=True,
    patience=3,
    plots=True
)

Ultralytics 8.4.184  Python-3.11.9 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\yatha\Desktop\autonomous-driving-perception\data\kitti.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=3, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=kitti_smoke_